# Detecting Product Cannibalization Through Changes in Product Nutritional Profiles
#### By Mattheus Marcus Contreras and Nikkei Jane Espina


In [ ]:
import pandas as pd
import sqlite3

transactions = pd.read_csv("data/transaction_data.csv")
products = pd.read_csv("data/product.csv")
household = pd.read_csv("data/hh_demographic.csv")
campaigns = pd.read_csv("data/campaign_table.csv")
coupons = pd.read_csv("data/coupon.csv")

conn = sqlite3.connect(":memory:")

transactions.to_sql("transactions", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")
household.to_sql("household", conn, index=False, if_exists="replace")
coupons.to_sql("coupons", conn, index=False, if_exists="replace")
campaigns.to_sql("campaigns", conn, index=False, if_exists="replace")

124548

In [ ]:
query = """
SELECT
    t.household_key,
    t.basket_id,
    t.day,
    t.product_id,
    t.quantity,
    t.sales_value,

    p.commodity_desc,
    p.sub_commodity_desc,
    p.brand,
    p.manufacturer,
    h.classification_1,
    h.classification_2,
    h.classification_3,
    h.classification_4,
    h.classification_5,
    h.homeowner_desc,
    h.kid_category_desc,
FROM transactions t
LEFT JOIN products p
    ON t.product_id = p.product_id
LEFT JOIN household h
    ON t.household_key = h.household_key
"""

master_transactions = pd.read_sql_query(query, conn)

master_transactions

####transactions - 2,595,732
#####products - 92,353
##household - 801
###coupons - 124,548

In [ ]:
query = """  

"""
master_promotions = pd.read_sql_query(query, conn)
master_promotions


In [2]:
sales_query = """
SELECT
    p.COMMODITY_DESC AS category,
    SUM(t.SALES_VALUE) AS total_sales,
    SUM(t.QUANTITY) AS total_quantity,
    COUNT(DISTINCT t.BASKET_ID) AS basket_count
FROM transactions t
JOIN products p
    ON t.PRODUCT_ID = p.PRODUCT_ID
GROUP BY p.COMMODITY_DESC
"""

sales_df = pd.read_sql_query(sales_query, conn)

sales_df

,category,total_sales,total_quantity,basket_count
0,,0.00,0,7392
1,(CORP USE ONLY),283.40,76,61
2,ADULT INCONTINENCE,4995.97,638,509
3,AIR CARE,17699.31,6861,3825
4,ANALGESICS,25426.21,5528,4258
...,...,...,...,...
303,WAREHOUSE SNACKS,15848.49,10633,6554
304,WATCHES/CALCULATORS/LOBBY,258.75,49,42
305,WATER,3904.39,3672,1425
306,WATER - CARBONATED/FLVRD DRINK,63873.10,30061,18019


In [ ]:
quality_query = """
SELECT 
    fn.fdc_id,
    AVG(CASE WHEN LOWER(n.name) LIKE '%protein%' THEN fn.amount
        END) AS avg_protein,
    AVG(CASE WHEN LOWER(n.name) LIKE '%sugar%' THEN fn.amount
        END) AS avg_sugar,
    AVG(CASE WHEN LOWER(n.name) LIKE '%fiber%' OR LOWER(n.name) LIKE '%fibre%' THEN fn.amount
        END) AS avg_fiber,
    AVG(
        CASE
            WHEN LOWER(n.name) LIKE '%sodium%'
            THEN fn.amount
        END
    ) AS avg_sodium
FROM food_nutrient fn
JOIN nutrient n
    ON fn.nutrient_id = n.id
GROUP BY fn.fdc_id
"""
quality_df = pd.read_sql_query(quality_query, conn)

quality_df

,fdc_id,avg_protein,avg_sugar,avg_fiber,avg_sodium
0,344604,0.81,2.440,0.266667,203.0
1,344605,0.81,2.440,0.266667,203.0
2,344606,23.21,0.000,0.000000,67.0
3,344607,23.21,NaN,0.000000,67.0
4,344608,18.75,NaN,0.000000,103.0
...,...,...,...,...,...
1906507,2779235,0.79,2.785,0.800000,476.0
1906508,2779236,0.79,2.785,0.800000,381.0
1906509,2779237,0.80,1.600,0.800000,480.0
1906510,2779238,5.71,0.000,8.600000,886.0


In [16]:
quality_query = """
SELECT
    fn.fdc_id,
    AVG(CASE WHEN LOWER(n.name) LIKE '%protein%' THEN fn.amount
        END) AS avg_protein,
    AVG(CASEWHEN LOWER(n.name) LIKE '%sugar%'
            THEN fn.amount
        END
    ) AS avg_sugar,

    AVG(
        CASE
            WHEN LOWER(n.name) LIKE '%fiber%'
              OR LOWER(n.name) LIKE '%fibre%'
            THEN fn.amount
        END
    ) AS avg_fiber,

    AVG(
        CASE
            WHEN LOWER(n.name) LIKE '%sodium%'
            THEN fn.amount
        END
    ) AS avg_sodium

FROM food_nutrient fn
JOIN nutrient n
    ON fn.nutrient_id = n.id
"""

quality_df = pd.read_sql_query(
    quality_query,
    conn
)

quality_df 

,fdc_id,avg_protein,avg_sugar,avg_fiber,avg_sodium
0,1105904,7.625234,16.785161,2.925287,1509.655846


In [18]:
from scipy.stats import zscore

quality_df["degradation_score"] = (
      zscore(quality_df["avg_sugar"].fillna(0))
    + zscore(quality_df["avg_sodium"].fillna(0))
    - zscore(quality_df["avg_protein"].fillna(0))
    - zscore(quality_df["avg_fiber"].fillna(0))
)

print("Sales Dataset Shape:", sales_df.shape)
print("Quality Dataset Shape:", quality_df.shape)

quality_df.head()

Sales Dataset Shape: (308, 4)
Quality Dataset Shape: (1906512, 6)


,fdc_id,avg_protein,avg_sugar,avg_fiber,avg_sodium,degradation_score
0,344604,0.81,2.44,0.266667,203.0,0.305399
1,344605,0.81,2.44,0.266667,203.0,0.305399
2,344606,23.21,0.00,0.000000,67.0,-1.077754
3,344607,23.21,NaN,0.000000,67.0,-1.077754
4,344608,18.75,NaN,0.000000,103.0,-0.809466


In [19]:
print("Sales Dataset Shape:", sales_df.shape)
print("Quality Dataset Shape:", quality_df.shape)

quality_df.head()

Sales Dataset Shape: (308, 4)
Quality Dataset Shape: (1906512, 6)


,fdc_id,avg_protein,avg_sugar,avg_fiber,avg_sodium,degradation_score
0,344604,0.81,2.44,0.266667,203.0,0.305399
1,344605,0.81,2.44,0.266667,203.0,0.305399
2,344606,23.21,0.00,0.000000,67.0,-1.077754
3,344607,23.21,NaN,0.000000,67.0,-1.077754
4,344608,18.75,NaN,0.000000,103.0,-0.809466
